# Merging Price & Resale Data with Applicants and Adding Missing Prices

This notebook performs a **two-step integration**:

1. Merge the **original merged price + resale dataset** (`merged_dataset_price&resale_Sept25.csv`)  
   with the **cleaned CHAPA applicant dataset** (`CHAPA_Chapter-40B_Application-Data_2021-2025_merged_v0.3.csv`), creating a dataset linking properties to applicants.

2. Add **completed missing prices** from `Property_Data_Jun2021_Sep2025_MissingResaleValues_Completed (1).csv` to the merged dataset, filling previously missing `Maximum Resale Price` values.

The final output is:  
`new_merged_dataset_filled.csv`

---

### Reference
- Price + Resale Dataset: `merged_dataset_price&resale_Sept25.csv`  
- Applicant Dataset: `CHAPA_Chapter-40B_Application-Data_2021-2025_merged_v0.3.csv`  
- Completed Price Dataset: `Property_Data_Jun2021_Sep2025_MissingResaleValues_Completed (1).csv`


In [ ]:
# ============================================================
# CHAPA Data Integration Pipeline
# Step: Merge Properties with Applicants & Add Missing Prices
# ============================================================

import pandas as pd

# ------------------------------------------------------------
# 1. Load Datasets
# ------------------------------------------------------------
def load_datasets():
    """Load all relevant datasets."""
    properties_df = pd.read_csv("/content/merged_dataset_price&resale_Sept25.csv")
    applicants_df = pd.read_csv("/content/CHAPA_Chapter-40B_Application-Data_2021-2025_merged_v0.3.csv")
    missing_prices_df = pd.read_csv("/content/Property_Data_Jun2021_Sep2025_MissingResaleValues_Completed (1).csv")
    return properties_df, applicants_df, missing_prices_df

# ------------------------------------------------------------
# 2. Standardize Columns for Merging
# ------------------------------------------------------------
def standardize_columns(properties_df, applicants_df):
    """Standardize Town, Address, and Unit columns to lowercase stripped strings."""
    for df, town_col, addr_col, unit_col in [
        (properties_df, 'Town', 'Address', 'Unit Number'),
        (applicants_df, 'property_town_city', 'property_street_address', 'property_unit')
    ]:
        df['Town_clean'] = df[town_col].str.strip().str.lower()
        df['Address_clean'] = df[addr_col].str.strip().str.lower()
        df['Unit_clean'] = df[unit_col].astype(str).str.strip().str.lower()
    return properties_df, applicants_df

# ------------------------------------------------------------
# 3. Merge Properties with Applicants
# ------------------------------------------------------------
def merge_properties_applicants(properties_df, applicants_df):
    """Left join applicant data to properties dataset."""
    merged = pd.merge(
        properties_df,
        applicants_df,
        on=['Town_clean', 'Address_clean', 'Unit_clean'],
        how='left',
        suffixes=('', '_applicant')
    )
    # Drop helper columns
    merged = merged.drop(columns=['Town_clean', 'Address_clean', 'Unit_clean'])
    return merged

# ------------------------------------------------------------
# 4. Add Missing Prices
# ------------------------------------------------------------
def add_missing_prices(merged_df, missing_prices_df):
    """Merge missing price data and fill missing Maximum Resale Price."""
    missing_prices_df = missing_prices_df.rename(columns={"Price": "Maximum Resale Price"})
    keys = ["Town", "Address", "Unit Number"]

    merged_updated = merged_df.merge(
        missing_prices_df[keys + ["Maximum Resale Price"]],
        on=keys,
        how='left',
        suffixes=("", "_new")
    )

    # Fill missing prices
    merged_updated["Maximum Resale Price"] = merged_updated["Maximum Resale Price"].fillna(
        merged_updated["Maximum Resale Price_new"]
    )

    # Drop helper column
    merged_updated.drop(columns=["Maximum Resale Price_new"], inplace=True)
    return merged_updated

# ------------------------------------------------------------
# 5. Save Dataset
# ------------------------------------------------------------
def save_dataset(df, filename="new_merged_dataset_filled.csv"):
    """Save final dataset to CSV."""
    df.to_csv(filename, index=False)
    print(f"✅ Dataset saved as {filename}")

# ------------------------------------------------------------
# 6. Pipeline Execution
# ------------------------------------------------------------
def main():
    print("🚀 Starting full CHAPA data integration pipeline...")
    properties_df, applicants_df, missing_prices_df = load_datasets()
    properties_df, applicants_df = standardize_columns(properties_df, applicants_df)
    merged_df = merge_properties_applicants(properties_df, applicants_df)
    final_df = add_missing_prices(merged_df, missing_prices_df)
    save_dataset(final_df)
    print("🏁 Pipeline complete. Dataset ready for analysis.")

if __name__ == "__main__":
    main()
